## Task 2: Hypothesis Test
### Hypothesis 1:
- On 1-type roads, the probability of dying in accidents was the same as on 3-type roads 
  
Import libraries

In [1]:
import pandas as pd
import scipy.stats as stats

from scipy.stats import chi2_contingency
from scipy.stats import f_oneway

Load the data

In [2]:
df = pd.read_pickle("accidents.pkl.gz")

The chances of fatal accidents on type 1 roads were similar to those on type 3 roads. I'll apply the 𝜒2 (chi-squared) test to see if this is true.
After that, I'll figure out if deadly accidents are more common on type 1 roads compared to type 3 roads. For this, I'll use the expected values from the 𝜒2 test results.

Select only accidents on type 1 and type 3 roads

In [3]:
data = df[(df['p36'].isin([1,3]))]

Create a new column 'lethal' to mark accidents with fatalities (p13a > 0)

In [4]:
data = df[(df['p36'].isin([1, 3]))].copy()
data.loc[:, 'death'] = data['p13a'] > 0

Create a contingency table with road types and lethality of accidents

In [5]:
tab = pd.crosstab(data['p36'], data['death'])

Perform Chi-squared test to compare the fatality rates on type 1 and type 3 roads

In [6]:
stat, p, dof, expected = stats.chi2_contingency(tab)

Set the significance level

In [7]:
alpha = 0.05
print('P-value:', p)

P-value: 2.95835646229767e-38


Determine if the fatality rates are significantly different

In [8]:
if p > alpha:
    print('Fatalities occur with similar frequencies on both road types.')
else:
    print('Fatalities occur with different frequencies on road types.')
    
    # Compare observed and expected frequencies to determine which type of road has more or less fatal accidents
    for i, (islethal, assumption) in enumerate(zip(tab.columns, (tab > expected).iloc[0].tolist()), start=1):
        frequency = ["more often", "less often"][not assumption]
        islethal = ["lethal", "non-lethal"][not islethal]
        print(f'On type {i} roads, {islethal} accidents occur {frequency}.')

Fatalities occur with different frequencies on road types.
On type 1 roads, non-lethal accidents occur less often.
On type 2 roads, lethal accidents occur more often.


### Conclusion
At a 95% confidence level, it's clear that the results are different between the two groups. This means that the chance of dying in accidents is not the same on type 1 roads as it is on type 3 roads.

### Hypothesis 2:
- In accidents involving Škoda brand vehicles, the damage to the vehicle is lower than in accidents involving Audi brand vehicles

Filter for only Škoda (39) and Audi (2) vehicles

In [9]:
data = df[df['p45a'].isin([39, 2])]

Separate the damage data for each brand

In [10]:
damage_skoda = data[data['p45a'] == 39]['p53']
damage_audi = data[data['p45a'] == 2]['p53']

Perform an independent t-test

In [11]:
test_statistic, p_value = stats.ttest_ind(damage_skoda, damage_audi, equal_var=False)

Define the significance level

In [12]:
alpha = 0.05
is_significant = p_value < alpha

Interpret the results

In [13]:
result_interpretation = "significantly different" if is_significant else "not significantly different"
damage_comparison = "less" if test_statistic < 0 else "more"

Conclusion

In [14]:
print(f"At a 95% confidence level, we find that the average damages for Škoda and Audi vehicles are {result_interpretation}.")
print(f"This suggests that the damage to Škoda vehicles is typically {damage_comparison} compared to Audi vehicles.")

At a 95% confidence level, we find that the average damages for Škoda and Audi vehicles are significantly different.
This suggests that the damage to Škoda vehicles is typically less compared to Audi vehicles.
